<a href="https://colab.research.google.com/github/Ram5268/ECON3916-Statistical-Machine-Learning/blob/main/Econ_3916_Assignment_3_Causal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
np.random.seed(42)

zeros = np.zeros(100)
tips = np.random.exponential(scale=5.0, size=150)

driver_tips = np.concatenate([zeros, tips])

print("Observerd median tip:", np.median(driver_tips))

# Manual Engine
B = 10000
n = len(driver_tips)

bootstrap_medians = np.empty(B)

for b in range (B):
  sample_indices = np.random.choice(n, size=n, replace=True)
  sample = driver_tips[sample_indices]
  bootstrap_medians[b] = np.median(sample)

ci_lower, ci_upper = np.percentile(bootstrap_medians, [2.5, 97.5])

print("95% Bootstrap CI for Median:", (ci_lower, ci_upper))

Observerd median tip: 0.7553316913699188
95% Bootstrap CI for Median: (np.float64(0.2653018357387816), np.float64(1.3635639228066991))


In [15]:
np.random.seed(42)

control = np.random.normal(loc=35, scale=5, size=500)
treatment = np.random.normal(loc=30, scale=5, size=500) # Defined treatment as a separate sample

observed_diff = np.mean(control) - np.mean(treatment)

print("Observed Difference (Control - Treatment):", observed_diff) # Corrected typo 'Obersverd'

combined = np.concatenate([control, treatment])

R = 5000
perm_diffs = np.empty(R)

for r in range(R):
  shuffled = np.random.permutation(combined)
  pseudo_control = shuffled[:500]
  pseudo_treatment = shuffled[500:]

  perm_diffs[r] = np.mean(pseudo_control) - np.mean(pseudo_treatment)

p_value = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))

print("Permutation Test P-value:", p_value)

Observed Difference (Control - Treatment): 4.875059387663217
Permutation Test P-value: 0.0


In [16]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

#Loading the dataset
df = pd.read_csv("swiftcart_loyalty.csv")

# Doublel check
print(df.head())
print(df.columns)
print(df.shape)

#Naive Simple Difference in Means

subscribers = df[df["subscriber"] == 1]
non_subscribers = df[df["subscriber"] == 0]

naive_sdo = subscribers ["post_spend"].mean() - non_subscribers ["post_spend"].mean()

print("Mean post_spend for Subscribers:", subscribers ["post_spend"].mean ())
print("Mean post_spend for Non-Subscribers:", non_subscribers["post_spend"].mean())
print("Naive Simple Difference in Outcomes (SD0):", naive_sdo)

#Propensity Score Model

X = df [["pre_spend","account_age", "support_tickets"]]
y = df ["subscriber"]

logit = LogisticRegression(max_iter=1000)
logit.fit(X,y)

#Predicted propensity score = P(subscriber = 1 | X)
df["propensity_score"] = logit.predict_proba(X) [:, 1]

print(df[["subscriber", "propensity_score"]].head())

# Each Subscriber matched to nearest non subscriber

treated = df[df["subscriber"] == 1].copy()
control = df[df["subscriber"] == 0].copy()

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[["propensity_score"]])

distances, indices = nn.kneighbors(treated[["propensity_score"]])

matched_control = control.iloc[indices.flatten()].copy()
matched_control.index = treated.index # align rows for comparison

#ATT: Average Treatment Effect on the Treated

att = np.mean(treated["post_spend"].values - matched_control["post_spend"].values)

print("Naive SD0", naive_sdo)
print("ATT after Propensity Score Matching", att)

   subscriber  pre_spend  account_age  support_tickets  post_spend
0           1  57.450712           37                2   85.169648
1           1  47.926035           41                0   72.802404
2           1  59.715328           41                0   79.858905
3           1  72.845448           34                0   80.335466
4           1  46.487699           34                2   67.956227
Index(['subscriber', 'pre_spend', 'account_age', 'support_tickets',
       'post_spend'],
      dtype='object')
(8941, 5)
Mean post_spend for Subscribers: 74.04358604052543
Mean post_spend for Non-Subscribers: 56.47291665600164
Naive Simple Difference in Outcomes (SD0): 17.57066938452379
   subscriber  propensity_score
0           1          0.546500
1           1          0.548460
2           1          0.683988
3           1          0.779637
4           1          0.397513
Naive SD0 17.57066938452379
ATT after Propensity Score Matching 9.913855182824864
